# Chapter 11 — Satellite Communications — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


In [ ]:
# Prerequisite from Ch 4 (so this chapter's notebook runs standalone):
def fspl_db(d_m, f_hz):
    return 20*np.log10(4*np.pi*np.asarray(d_m, float)*f_hz / C)

## 20. Satellite Links — Ch 11  *(future, outdoor)*

The final chapter, and a capstone that **reuses everything**: FSPL (§1), the rain model (§19), and
the noise floor (§10). Skip for the indoor engine (single long slant path through the whole
atmosphere), but the geometry and hot-pad noise are clean and worth having.

- **Slant-range geometry:** central angle ψ from lat/long (eq 11.4), slant range `rs` (eq 11.5),
  elevation `θ` (eq 11.6). GEO radius ≈ 42,242 km from earth center. → `sat_central_angle()`,
  `sat_slant_range_km()`, `sat_elevation_deg()`. *Verified Ex 11.1 (rs=36,314 km, θ=66.6°).*
- **ITU satellite rain** (P.618 10-step, eqs 11.9-11.25) extends §19 with a rain-cell-height
  geometry. → `itu_sat_rain_atten_db()`. *Verified Ex 11.3 uplink (A0.01=38.5 dB, A0.1=14.8 dB).*
- **Noise (absorptive losses raise the noise floor):** hot-pad `TN = (Tin + (L−1)T)/L` (eq 11.50),
  rain temp `Tr = Tm(1 − 10^(−A/10))` (eq 11.51), figure of merit `G/T`. → `hotpad_noise_temp()`,
  `rain_noise_temp()`, `gt_db()`. *Verified Ex 11.5/11.6 (TN=222.5 K, Tr=204.4 K).*
- **Doppler** `fd = vr·f0/c` (eq 11.1) = §16 `doppler_shift_hz`. GEO round-trip delay ≈ 240 ms.
  Ionospheric effects (Faraday rotation, scintillation) only matter <10 GHz → circular pol preferred.


In [ ]:
def sat_central_angle(Le, le, Ls, ls):
    # Central angle psi (deg) from earth-station & subsatellite lat/long (deg). Eq 11.4.
    Le, le, Ls, ls = map(np.radians, (Le, le, Ls, ls))
    return np.degrees(np.arccos(np.cos(Le)*np.cos(Ls)*np.cos(ls-le) + np.sin(Le)*np.sin(Ls)))

def sat_slant_range_km(psi_deg, h_km=42242.0, re_km=6378.0):
    p = np.radians(psi_deg)
    return h_km*np.sqrt(1 + (re_km/h_km)**2 - 2*(re_km/h_km)*np.cos(p))     # eq 11.5

def sat_elevation_deg(psi_deg, rs_km, h_km=42242.0):
    return np.degrees(np.arccos(h_km*np.sin(np.radians(psi_deg))/rs_km))    # eq 11.6

# Example 11.1: GEO sat, earth station at 20 deg latitude, same longitude
rs = sat_slant_range_km(20)
print(f"Ex 11.1: slant range={rs:.0f} km (book 36,314), elevation={sat_elevation_deg(20, rs):.1f} deg (book 66.6)")
print(f"         FSPL to that GEO sat @12 GHz = {fspl_db(rs*1e3, 12e9):.1f} dB")


In [ ]:
def itu_sat_rain_atten_db(freq_ghz, elev_deg, lat_deg, RR001, k, a, hs_km=0.0, availability_pct=99.99):
    # ITU-R P.618 slant-path rain fade (10-step), returns fade depth (dB) at the given availability.
    th = elev_deg; st = np.sin(np.radians(th))
    hR = 4.0 if abs(lat_deg) < 36 else 4.0 - 0.075*(abs(lat_deg) - 36)      # step 1
    if th < 5:                                                             # step 2
        Lsl = 2*(hR-hs_km)/(np.sqrt(st**2 + 2*(hR-hs_km)/8500) + st)
    else:
        Lsl = (hR - hs_km)/st
    LG = Lsl*np.cos(np.radians(th))                                        # step 3
    gR = k*RR001**a                                                       # step 5
    r001 = 1/(1 + 0.78*np.sqrt(LG*gR/freq_ghz) - 0.38*(1 - np.exp(-2*LG))) # step 6
    zeta = np.degrees(np.arctan((hR - hs_km)/(LG*r001)))                   # step 7
    LR = LG*r001/np.cos(np.radians(th)) if zeta > th else (hR - hs_km)/st
    chi = 36 - abs(lat_deg) if abs(lat_deg) < 36 else 0.0
    v001 = 1/(1 + np.sqrt(st)*(31*(1 - np.exp(-th/(1+chi)))*np.sqrt(LR*gR)/freq_ghz**2 - 0.45))
    A001 = gR*(LR*v001)                                                    # steps 8-9
    p = 100 - availability_pct                                            # step 10
    if p >= 1 or abs(lat_deg) >= 36:      b = 0.0
    elif th > 25:                          b = -0.005*(abs(lat_deg) - 36)
    else:                                  b = -0.005*(abs(lat_deg) - 36) + 1.8 - 4.25*st
    exp = -(0.655 + 0.033*np.log(p) - 0.045*np.log(A001) - b*(1-p)*st)
    return A001*(p/0.01)**exp

# Example 11.3 uplink: 30 GHz, NYC (region K, 42 mm/h, lat 40), circular -> k=0.177, a=1.011, elev 40.9
A001 = itu_sat_rain_atten_db(30, 40.9, 40, 42, 0.177, 1.011, availability_pct=99.99)
A01  = itu_sat_rain_atten_db(30, 40.9, 40, 42, 0.177, 1.011, availability_pct=99.9)
print(f"Ex 11.3 uplink: A(0.01%)={A001:.1f} dB (book 38.5), A(0.1%)={A01:.1f} dB (book 14.8)")


In [ ]:
def hotpad_noise_temp(Tin_K, L_db, T_atten_K=290.0):
    # Attenuator (loss L) raises the noise temp: TN = (Tin + (L-1) T)/L. Eq 11.50.
    L = 10**(L_db/10.0)
    return (Tin_K + (L - 1)*T_atten_K)/L

def rain_noise_temp(A_db, Tm_K=273.0):
    return Tm_K*(1 - 10**(-A_db/10.0))                    # eq 11.51 (added to Tsys)

def gt_db(G_db, Tsys_K):
    return G_db - 10*np.log10(Tsys_K)                     # figure of merit G/T (dB/K)

# Example 11.5 (hot-pad) & 11.6 (rain temp): Ta=20 K, Te=50 K, 6 dB rain fade
Ta, Te, fade = 20.0, 50.0, 6.0
TN   = hotpad_noise_temp(Ta, fade)                        # hot-pad
Tsys_rain_hp = TN + Te
Tr   = rain_noise_temp(fade)                              # rain-temperature method
Tsys_rain_rt = Ta + Tr + Te
print(f"Ex 11.5 hot-pad: TN={TN:.1f} K, Tsys_rain={Tsys_rain_hp:.1f} K (book 222.5 / 272.5)")
print(f"Ex 11.6 rain-T : Tr={Tr:.1f} K, Tsys_rain={Tsys_rain_rt:.1f} K (book 204.4 / 274.4)")
print(f"  noise up {10*np.log10(Tsys_rain_hp/(Ta+Te)):.1f} dB + signal down {fade:.0f} dB = "
      f"{10*np.log10(Tsys_rain_hp/(Ta+Te))+fade:.1f} dB SNR loss (book Ex11.5 says 9.9 = typo; Ex11.6 says 11.9)")
print(f"  G/T (G=30 dBi, Tsys=70 K) = {gt_db(30, 70):.1f} dB/K (book prints 14.3 = the linear ratio 1000/70, mislabeled)")
